In [ ]:
# @title Upload original Conquer Mobile assets from Google Drive to this game's GitHub Release
# One-cell workflow: mount Drive -> find archive -> verify size/SHA-256 -> create draft Release -> upload.
# Nothing is added to the Git repository itself; the large ZIP is uploaded as a Release asset.

from google.colab import drive
from pathlib import Path
from getpass import getpass
import hashlib, requests, sys, os
from urllib.parse import quote

REPO = "ZzwahwahzZ/Clash-of-Legends"
BRANCH = "tools/colab-drive-asset-transfer"
TAG = "original-mobile-assets-2026-11-06"
FILENAME = "extracted-assets-001.zip"
EXPECTED_SIZE = 842_723_106  # size recorded in the earlier project audit

print("STEP 1/6 — Connect Google Drive")
drive.mount("/content/drive")

print("\nSTEP 2/6 — Locate the original archive")
roots = [Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")]
matches = []
for root in roots:
    if root.exists():
        matches.extend(p for p in root.rglob(FILENAME) if p.is_file())

# De-duplicate paths in case mount points overlap.
matches = list(dict.fromkeys(matches))
if not matches:
    raise FileNotFoundError(
        f"Could not find {FILENAME} in MyDrive or Shared drives. "
        "Check the exact filename and make sure the Drive containing it is mounted."
    )

if len(matches) > 1:
    print("More than one matching archive was found:")
    for i, p in enumerate(matches):
        print(f"  [{i}] {p} ({p.stat().st_size:,} bytes)")
    choice = int(input("Enter the number of the correct original archive: ").strip())
    if choice < 0 or choice >= len(matches):
        raise ValueError("Invalid selection.")
    source = matches[choice]
else:
    source = matches[0]

size = source.stat().st_size
print(f"Selected: {source}")
print(f"Size: {size:,} bytes ({size / 1024**2:.2f} MiB)")
if size != EXPECTED_SIZE:
    raise ValueError(
        f"Size mismatch: expected {EXPECTED_SIZE:,} bytes from the project audit, "
        f"but found {size:,}. No upload was attempted. Check that this is the original archive."
    )

print("\nSTEP 3/6 — Calculate SHA-256 (reads the archive without extracting it)")
digest = hashlib.sha256()
with source.open("rb") as f:
    while True:
        block = f.read(16 * 1024 * 1024)
        if not block:
            break
        digest.update(block)
sha256 = digest.hexdigest()
print("Local SHA-256:", sha256)

print("\nSTEP 4/6 — Authenticate to GitHub")
token = getpass("Paste a GitHub token with Contents: Read and write for this repository: ").strip()
if not token:
    raise ValueError("No GitHub token supplied.")

api = "https://api.github.com"
headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}
repo_check = requests.get(f"{api}/repos/{REPO}", headers=headers, timeout=30)
repo_check.raise_for_status()
permissions = repo_check.json().get("permissions", {})
if not permissions.get("push", False):
    raise PermissionError("This token/account does not have push access to the target repository.")

print("\nSTEP 5/6 — Create or reuse a draft Release")
release_url = f"{api}/repos/{REPO}/releases/tags/{TAG}"
check = requests.get(release_url, headers=headers, timeout=30)

if check.status_code == 200:
    release = check.json()
elif check.status_code == 404:
    created = requests.post(
        f"{api}/repos/{REPO}/releases",
        headers=headers,
        json={
            "tag_name": TAG,
            "target_commitish": BRANCH,
            "name": "Original Conquer Mobile Resource Archive",
            "body": (
                "Original resource archive for Clash of Legends asset forensics. "
                f"Source filename: {FILENAME}\n"
                f"Expected source size: {size} bytes\n"
                f"Local SHA-256: {sha256}\n"
                "Uploaded from the user's Google Drive through the project Colab notebook."
            ),
            "draft": True,
            "prerelease": True
        },
        timeout=60
    )
    created.raise_for_status()
    release = created.json()
else:
    check.raise_for_status()

existing = [a for a in release.get("assets", []) if a.get("name") == FILENAME]
if existing:
    asset = existing[0]
    raise RuntimeError(
        f"A release asset named {FILENAME} already exists ({asset.get('size')} bytes). "
        "Nothing was overwritten. Inspect that asset before deciding whether to replace it."
    )

upload_url = release["upload_url"].split("{", 1)[0] + "?name=" + quote(FILENAME)
upload_headers = {
    **headers,
    "Content-Type": "application/zip",
    "Content-Length": str(size)
}

print(f"Uploading {size:,} bytes to Release {TAG}. Keep this Colab session running...")
with source.open("rb") as f:
    uploaded = requests.post(
        upload_url,
        headers=upload_headers,
        data=f,
        timeout=(60, 7200)
    )
uploaded.raise_for_status()
asset = uploaded.json()

print("\nSTEP 6/6 — Verify GitHub's uploaded asset metadata")
if asset.get("name") != FILENAME or asset.get("size") != size:
    raise RuntimeError(
        "GitHub returned asset metadata that does not match the source file. "
        f"Returned name={asset.get('name')!r}, size={asset.get('size')!r}."
    )

print("\nUPLOAD COMPLETED")
print("Release:", release.get("html_url"))
print("Asset:", asset.get("browser_download_url"))
print("Filename:", asset.get("name"))
print("Bytes:", f"{asset.get('size'):,}")
print("Source SHA-256:", sha256)
print("\nCopy the full report above and send it back in the project chat. "
      "The asset URL and SHA-256 will be used for the next extraction and integrity checks.")
